In [1]:
import os
from pathlib import Path
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter

from langchain_classic.storage import LocalFileStore
from langchain_classic.storage import create_kv_docstore

In [2]:
# local embedding model import
MODEL_PATH = Path.home() / "rag-learning-practice" / "models" / "all-MiniLM-L6-v2"

# Local embedding model setup
embeddings = HuggingFaceEmbeddings(
    model_name=str(MODEL_PATH),
    model_kwargs={
        "device": "cpu",
        "local_files_only": True
    },
    encode_kwargs={
        "normalize_embeddings": True
    }
)


/home/mehedi-hassan/rag-learning-practice/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4409.29it/s]


In [3]:

# এটিকে একটি বড় document ধরছি
parent_document = Document(
    page_content=(
        "Retrieval Augmented Generation, or RAG, is a technique used in "
        "LLM applications. In RAG, relevant documents are retrieved from "
        "an external knowledge base and provided to the language model as "
        "context. This helps the model answer questions using up-to-date "
        "or private information. A retriever finds relevant documents, "
        "while a vector store stores embeddings for similarity search."
    ),
    metadata={"source": "rag_guide"}
)

# ছোট child chunk বানানোর জন্য splitter
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=50,
    chunk_overlap=10
)

# Parent document store, যেখানে পুরো document থাকবে
docstore = create_kv_docstore(LocalFileStore("../all_vector_stores/parent_docstore"))

# Child chunk-এর জন্য vector store
vector_store = Chroma(
    collection_name="parent_document_doc",
    embedding_function=embeddings,
    persist_directory="../all_vector_stores/chroma_parent_document_rectriever"
)


In [4]:
# ParentDocumentRetriever তৈরি করছি
retriever = ParentDocumentRetriever(
    vectorstore=vector_store,
    child_splitter=child_splitter,  # ছোট chunk দিয়ে search হবে
    docstore=docstore,              # parent document এখানে থাকবে
    search_kwargs={"k": 2}          # search করার পর কতগুলো parent document return হবে
)

In [5]:
# Parent document add করছি, ভিতরে এটি child chunk বানিয়ে vector store-এ রাখবে
retriever.add_documents([parent_document])

query = "What does a retriever do in RAG?"

# এখানে result হবে ছোট chunk নয়, বরং পুরো parent document
results = retriever.invoke(query)

print("Query:", query)
print("-" * 50)

for i, doc in enumerate(results, start=1):
    print(f"{i}. {doc.page_content}")

Query: What does a retriever do in RAG?
--------------------------------------------------
1. Retrieval Augmented Generation, or RAG, is a technique used in LLM applications. In RAG, relevant documents are retrieved from an external knowledge base and provided to the language model as context. This helps the model answer questions using up-to-date or private information. A retriever finds relevant documents, while a vector store stores embeddings for similarity search.
